# Prueba final con los sujetos reservados 13 y 14

**Objetivo:** medir el modelo final en dos personas que **no se usaron en ninguna decisión del proyecto**. Desde el avance 4, los sujetos 13 y 14 quedaron fuera de toda la validación cruzada (LOSO con los sujetos 1–12). Este notebook es la única vez que se usan.

**Modelos que se evalúan:**

| Modelo | Configuración (la del notebook 11) |
|---|---|
| **CPC-GRU `EML`** | encoder causal (32/64/128) + GRU 2 × 256; CPC sin etiquetas con horizontes de 1 a 12 pasos; cabeza lineal; LP-FT |
| **Random Forest `EML`** | 300 árboles; 5 estadísticas por variable (media, desviación, mínimo, máximo, pendiente) → 50 características |

Los dos usan las 10 variables de `EML` (codos, manos y piernas; 11 sensores) y ventanas de 4 s etiquetadas con el instante final.

**Dos escenarios:**

| Escenario | Entrenamiento | Validación (parada temprana) | Prueba | Para qué |
|---|---|---|---|---|
| **A. Prueba final** | sujetos 1–11 | sujeto 12 | **13 y 14, por separado** | la estimación honesta del modelo final |
| **B. Réplica del protocolo de Bassani et al. (2025)** | sujetos 1–11 y 13 | sujeto 12 | **14** | comparar con su "LOSO" (13 sujetos → sujeto 14): BiLSTM 0.903 |

- **Por qué 1–11 + 12 de validación:** es la misma estructura de cada ronda LOSO (un sujeto de validación interna y el resto para entrenar), ahora con 11 sujetos de entrenamiento en lugar de 10. El Random Forest no necesita validación, pero se entrena con los **mismos sujetos** que la red para que la comparación sea justa (como en el notebook 11).
- **El escenario B replica la partición de sujetos de Bassani, no su preprocesamiento:** ellos usan 1 s a 240 Hz, 8 variables de movimiento + 2 de sEMG y otro etiquetado. Aquí se conserva todo lo nuestro y sólo cambia qué sujetos entrenan y cuál se evalúa. Se corre **después** del escenario A porque usa el sujeto 13 para entrenar.
- **3 semillas** (0, 1, 2): cada una cambia la inicialización y el muestreo de CPC, el ajuste fino y el Random Forest. Se reporta la media entre semillas y el ensamble (promedio de las probabilidades de las 3).

**Reglas de la prueba final (declaradas antes de ejecutar):**

1. La configuración está congelada: es la del notebook 11 para `EML`, sin ningún cambio.
2. Los sujetos 13 y 14 no intervienen en el escalador, la parada temprana ni ninguna otra decisión.
3. El notebook se ejecuta **una sola vez**. Si algo falla, se corrige el error de código, no la configuración.
4. La métrica principal es el **F1 macro de cada sujeto de prueba**, como en LOSO.

# 1. Configuración

In [ ]:
import copy, json, os, re, time
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from numpy.lib.stride_tricks import sliding_window_view
from scipy.interpolate import interp1d
from scipy.signal import butter, sosfiltfilt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ----------------------------- Datos (iguales a los notebooks 10–17) -----------------------------
SEED = 0
TARGET_FS = 12                         # Hz
CV_SUBJECTS = tuple(range(1, 13))      # sujetos de desarrollo (LOSO)
TEST_SUBJECTS = (13, 14)               # reservados: se usan por primera vez aquí
WIN, SUP_HOP, CPC_HOP = 48, 6, 24      # ventana de 4 s; una ventana etiquetada cada 0.5 s y una sin etiqueta (CPC) cada 2 s
SCALER_FIT_POINTS = 250_000            # instantes que se muestrean para ajustar el escalador
PROBE_TRAIN_MAX = 15_000               # ventanas de la sonda que decide la parada temprana de CPC

# ----------------------------- CPC-GRU + LP-FT (los valores del notebook 11) -----------------------------
CPC_EPOCHS, CPC_EVAL_EVERY, CPC_PATIENCE = 20, 2, 3
CPC_BATCH, CPC_LR, GRAD_CLIP, N_ANCHORS = 128, 5e-4, 1.0, 4
HORIZONS = tuple(range(1, 13))         # CPC predice de 1 a 12 pasos adelante (1 s)
FINETUNE_BATCH = 256
FINETUNE_LR_HEAD, FINETUNE_LR_BACKBONE = 1e-3, 1e-4
FT_EPOCHS, FT_PATIENCE = 30, 5         # etapa FT de LP-FT
LP_EPOCHS, LP_PATIENCE, LP_LR = 20, 3, 1e-3   # etapa LP de LP-FT
STEPS_CAP = FT_TRAIN_CAP = None        # límites sólo para la corrida de prueba

# ----------------------------- Random Forest (el del notebook 11) -----------------------------
RF_PARAMS = dict(n_estimators=300, max_features="sqrt", min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1)

# ----------------------------- Escenarios y semillas -----------------------------
VAL_SUBJECT = 12
SCENARIOS = {
    "A_final": {"nombre": "A. Prueba final", "train": tuple(range(1, 12)), "val": VAL_SUBJECT, "test": (13, 14)},
    "B_bassani": {"nombre": "B. Réplica Bassani (1–13 → 14)", "train": tuple(range(1, 12)) + (13,), "val": VAL_SUBJECT,
                  "test": (14,)},
}
SEEDS = (0, 1, 2)

# Corrida de prueba rápida (1 semilla, pocas épocas) para revisar que todo funciona: NB18_QUICK=1
QUICK = os.environ.get("NB18_QUICK") == "1"
if QUICK:
    SEEDS = (0,)
    CPC_EPOCHS, CPC_EVAL_EVERY, STEPS_CAP = 2, 1, 5
    PROBE_TRAIN_MAX, FT_TRAIN_CAP, LP_EPOCHS, FT_EPOCHS = 2_000, 2_000, 1, 1
    RF_PARAMS["n_estimators"] = 20

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SYNC_DIR = REPO_DIR / "notebooks" / "_cache" / "sync" / f"fs{TARGET_FS}"                     # exportación MVNX sincronizada
RECOVER_DIR = REPO_DIR / "notebooks" / "_cache" / "sync" / f"fs{TARGET_FS}_mvnx_recuperados"  # caché de los ensayos recuperados
MVNX_DIR, MVNX_LABEL_DIR = REPO_DIR / "datos_unzipped" / "mvnx_files", REPO_DIR / "datos_unzipped" / "Labels 2"
NB11_DIR = REPO_DIR / "checkpoints" / f"nb11_variables_minimas_fs{TARGET_FS}"          # LOSO de referencia (EML)
RUN_DIR = REPO_DIR / "checkpoints" / (f"nb18_prueba_final_fs{TARGET_FS}" + ("_quick" if QUICK else ""))
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_DIR / "progreso.log"
assert (SYNC_DIR / "metadata.json").exists(), f"Falta la exportación sincronizada en {SYNC_DIR.relative_to(REPO_DIR)}"
assert (NB11_DIR / "gru_ft" / "EML").exists(), f"Faltan las rondas EML del notebook 11 en {NB11_DIR.relative_to(REPO_DIR)}"

SYNC_META = json.loads((SYNC_DIR / "metadata.json").read_text())
LABELS, NO_LABEL = SYNC_META["label_order"], SYNC_META["sin_etiqueta"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


def log(message):
    """Imprime un mensaje con hora y lo guarda en progreso.log (útil para seguir corridas largas)."""
    line = f"{datetime.now():%Y-%m-%d %H:%M} | {message}"
    print(line)
    with open(LOG_FILE, "a") as fh:
        fh.write(line + "\n")


print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE} | semillas: {SEEDS} | QUICK: {QUICK}")
print(f"Resultados en: {RUN_DIR.relative_to(REPO_DIR)}")

# 2. Datos

Los mismos de los notebooks 11–17: MVNX a 12 Hz, las 30 variables de Xsens, etiqueta del instante final, ventanas de 4 s y los 3 ensayos recuperados del sujeto 7. Las celdas son las del notebook 14 sin cambios, más las estadísticas por ventana del Random Forest (notebook 11).

In [ ]:
# Columnas MVNX crudas que se necesitan para calcular las 30 variables
RAW_COLUMNS = (
    [f"position__{s}__{a}" for s in ("Pelvis", "Head", "LeftHand", "RightHand") for a in "xyz"]          # posiciones (m)
    + [f"velocity__{s}__{a}" for s in ("Pelvis", "LeftHand", "RightHand") for a in "xyz"]               # velocidades (m/s)
    + [f"angularVelocity__{s}__{a}" for s in ("Pelvis", "T8") for a in "xyz"]                            # giros (rad/s)
    + [f"jointAngle__{j}__c3" for j in ("jL5S1", "jC1Head", "jLeftElbow", "jRightElbow", "jLeftWrist", "jRightWrist",
                                       "jLeftHip", "jRightHip", "jLeftKnee", "jRightKnee")]               # flexiones (°)
    + [f"jointAngleXZY__{j}__{c}" for j in ("jLeftShoulder", "jRightShoulder") for c in ("c1", "c3")]   # hombros en XZY
    + [f"footContacts__{f}_{p}" for f in ("LeftFoot", "RightFoot") for p in ("Heel", "Toe")]            # contactos (0/1)
)
RAW_IDX = {c: i for i, c in enumerate(RAW_COLUMNS)}   # nombre de columna -> posición
SIDES = (("izq", "Left"), ("der", "Right"))           # prefijo en español y nombre del lado en el MVNX


def derive(raw):
    """Columnas MVNX crudas -> las 30 variables, con los lados fijos (izquierdo / derecho)."""
    col = lambda name: raw[:, RAW_IDX[name]].astype(np.float64)
    # --- Postura (14): ángulos articulares en grados ---
    out = {"tronco_flexion": col("jointAngle__jL5S1__c3"), "cuello_flexion": col("jointAngle__jC1Head__c3")}
    for lado, s in SIDES:
        out[f"{lado}_hombro_flexion"] = col(f"jointAngleXZY__j{s}Shoulder__c3")
        out[f"{lado}_hombro_abduccion"] = col(f"jointAngleXZY__j{s}Shoulder__c1")
        out[f"{lado}_codo_flexion"] = col(f"jointAngle__j{s}Elbow__c3")
        out[f"{lado}_muneca_flexion"] = col(f"jointAngle__j{s}Wrist__c3")
        out[f"{lado}_cadera_flexion"] = col(f"jointAngle__j{s}Hip__c3")
        out[f"{lado}_rodilla_flexion"] = col(f"jointAngle__j{s}Knee__c3")
    # --- Alturas y alcance (6): en metros; el alcance es la distancia horizontal mano–pelvis ---
    out["pelvis_altura"] = col("position__Pelvis__z")
    out["cabeza_altura"] = col("position__Head__z")
    for lado, s in SIDES:
        out[f"{lado}_mano_altura"] = col(f"position__{s}Hand__z")
        out[f"{lado}_mano_alcance"] = np.hypot(col(f"position__{s}Hand__x") - col("position__Pelvis__x"),
                                               col(f"position__{s}Hand__y") - col("position__Pelvis__y"))
    # --- Velocidades (6): normas, para que no dependan de hacia dónde mira la persona ---
    out["pelvis_vel_horizontal"] = np.hypot(col("velocity__Pelvis__x"), col("velocity__Pelvis__y"))
    out["pelvis_vel_vertical"] = col("velocity__Pelvis__z")
    for lado, s in SIDES:
        out[f"{lado}_mano_rapidez"] = np.linalg.norm([col(f"velocity__{s}Hand__{a}") for a in "xyz"], axis=0)
    out["pelvis_giro"] = np.abs(col("angularVelocity__Pelvis__z"))
    out["tronco_vel_angular"] = np.linalg.norm([col(f"angularVelocity__T8__{a}") for a in "xyz"], axis=0)
    # --- Apoyo (4): contacto del talón y la punta de cada pie con el suelo ---
    for lado, s in SIDES:
        out[f"{lado}_talon"] = col(f"footContacts__{s}Foot_Heel")
        out[f"{lado}_punta"] = col(f"footContacts__{s}Foot_Toe")
    return np.column_stack(list(out.values())).astype(np.float32), list(out)


# Unidad y grupo de cada variable (se usan en tablas, en el escalador y en la gráfica de importancia)
def feature_unit(name):
    if name.endswith(("flexion", "abduccion")):
        return "°"
    if name.endswith(("altura", "alcance")):
        return "m"
    if name.endswith(("vel_horizontal", "vel_vertical", "rapidez")):
        return "m/s"
    if name.endswith(("giro", "vel_angular")):
        return "rad/s"
    return "0/1"

In [ ]:
# Ensayos recuperados: los que la exportación excluyó SÓLO por la sEMG se leen del MVNX crudo, con los mismos
# controles de calidad del MVNX y el mismo remuestreo que la exportación.
EXCLUDED = pd.read_csv(SYNC_DIR / "exclusiones.csv")
SEMG_ONLY = EXCLUDED.groupby("mvnx_file")["motivo"].agg(lambda m: m.str.startswith("sEMG").all())   # excluido sólo por sEMG
RECOVER = sorted(SEMG_ONLY[SEMG_ONLY].index)
NS = "{http://www.xsens.com/mvn/mvnx}"                 # espacio de nombres del XML de Xsens
# Campos que se leen y cómo se remuestrea cada uno
FIELDS = {"position": "continua", "velocity": "continua", "angularVelocity": "continua",
          "jointAngle": "angulo", "jointAngleXZY": "angulo", "footContacts": "binaria"}
# Nombre de archivo -> sujeto, tarea, lado, carga y condición
MVNX_RE = re.compile(r"Subj_(?P<subject>\d+)_(?P<task>Isokin|MMH_Bim|MMH_One)_(?:(?P<side>[LR])_)?"
                     r"(?P<load_kg>\d+)kg_(?P<condition>St|\d+bpm|Trial_\d)\.mvnx")


def read_mvnx_fields(path):
    """Tiempos (s) y campos de FIELDS de los frames normales de un MVNX, con los nombres de su cabecera."""
    names = {"segment": [], "joint": [], "contactDefinition": []}
    times, raw = [], {}
    # El XML pesa cientos de MB: se lee por eventos y cada frame se borra después de leerlo
    with open(path, "rb") as fh:
        frames = None
        for event, el in ET.iterparse(fh, events=("start", "end")):
            tag = el.tag[len(NS):] if el.tag.startswith(NS) else el.tag
            if event == "start" and tag == "frames":
                frames = el
            elif event == "end" and tag in names and el.get("label") and frames is None:
                names[tag].append(el.get("label"))                     # nombres de segmentos, articulaciones y contactos
            elif event == "end" and tag == "frame":
                if el.get("type") == "normal":                         # se ignoran los frames de calibración
                    times.append(float(el.get("time")))
                    for c in el:
                        f = c.tag[len(NS):]
                        if f in FIELDS:
                            raw.setdefault(f, []).append(c.text or "")
                frames.remove(el)
    n = len(times)
    data = {k: np.array(" ".join(v).split(), dtype=np.float64).reshape(n, -1) for k, v in raw.items()}
    t = np.asarray(times) / 1000.0
    return t - t[0], data, names


# Nombres de columna de un campo, con el mismo formato que la exportación sincronizada
def mvnx_columns(field, names):
    if field in ("jointAngle", "jointAngleXZY"):
        return [f"{field}__{j}__c{c}" for j in names["joint"] for c in (1, 2, 3)]
    if field == "footContacts":
        return [f"{field}__{c}" for c in names["contactDefinition"]]
    return [f"{field}__{s}__{a}" for s in names["segment"] for a in "xyz"]


# Pasa-bajas antialias de fase cero (Butterworth de orden 4 a 4.8 Hz) antes de bajar de 240 a 12 Hz
def lowpass(x, fs):
    sos = butter(SYNC_META["antialias"]["orden"], SYNC_META["antialias"]["corte_hz"], btype="low", fs=fs, output="sos")
    return sosfiltfilt(sos, x, axis=0)


# Vecino más cercano: para contactos (0/1) y etiquetas, que no se pueden interpolar
def nearest(t, values, grid):
    idx = np.clip(np.searchsorted(t, grid), 1, len(t) - 1)
    idx -= (grid - t[idx - 1]) < (t[idx] - grid)
    return values[idx]


# Remuestrea un campo a la rejilla de 12 Hz según su tipo
def resample_field(t, x, grid, fs, kind):
    if kind == "binaria":
        return nearest(t, x, grid)
    if kind == "angulo":
        x = np.rad2deg(np.unwrap(np.deg2rad(x), axis=0))              # evita saltos de 180° a −180° al filtrar
    y = interp1d(t, lowpass(x, fs), axis=0, assume_sorted=True)(grid)
    return (y + 180.0) % 360.0 - 180.0 if kind == "angulo" else y


# Etiquetas válidas: actividades conocidas, segmentos con inicio antes del fin y sin traslapes
def labels_ok(lab):
    gap = lab["Start_Frame"].shift(-1) - lab["End_Frame"]
    return bool(lab["Label"].isin(LABELS).all() and not ((lab["End_Frame"] < lab["Start_Frame"]).any() or (gap < 0).any()))


def process_mvnx(mvnx_file, grid=None):
    """MVNX crudo -> (columnas RAW_COLUMNS, label_mvnx, tiempos) a TARGET_FS Hz, más sus controles de calidad."""
    subj = f"Subj_{int(MVNX_RE.fullmatch(mvnx_file)['subject']):02d}"
    t, data, names = read_mvnx_fields(MVNX_DIR / subj / mvnx_file)
    lab = pd.read_csv(MVNX_LABEL_DIR / subj / mvnx_file.replace(".mvnx", "_mvnx_labels.csv"))
    fs = (len(t) - 1) / t[-1]
    # Controles de calidad: saltos de tiempo en la grabación y etiquetas consistentes
    checks = {"n_time_gaps": int((np.diff(t) * 1000 > 1000 / 240 + 1.1).sum()), "labels_ok": labels_ok(lab)}
    if grid is None:
        grid = np.arange(int(np.floor(t[-1] * TARGET_FS)) + 1) / TARGET_FS
    cols, blocks = [], []
    for field, kind in FIELDS.items():
        blocks.append(resample_field(t, data[field], grid, fs, kind))
        cols += mvnx_columns(field, names)
    pos = {c: i for i, c in enumerate(cols)}
    mv = np.concatenate(blocks, axis=1)[:, [pos[c] for c in RAW_COLUMNS]].astype(np.float32)
    # Etiqueta de cada frame a partir de los segmentos anotados (en frames de MVNX)
    y = np.full(len(t), NO_LABEL, dtype=np.int8)
    base = SYNC_META["base_indices_mvnx"]
    for s0, s1, name in lab[["Start_Frame", "End_Frame", "Label"]].itertuples(index=False):
        y[max(0, s0 - base):s1 - base + 1] = LABELS.index(name)
    return mv, nearest(t, y, grid), grid, checks


def recovered_trial(mvnx_file):
    """Ensayo recuperado desde su caché, o desde el MVNX crudo la primera vez."""
    RECOVER_DIR.mkdir(parents=True, exist_ok=True)
    out = RECOVER_DIR / mvnx_file.replace(".mvnx", ".npz")
    if out.exists() and out.with_suffix(".json").exists():
        with np.load(out) as z:
            return z["mvnx"], z["label_mvnx"], z["time_s"], json.loads(out.with_suffix(".json").read_text())
    mv, lab, grid, checks = process_mvnx(mvnx_file)
    checks["nan"] = int(np.isnan(mv).sum())
    np.savez_compressed(out, mvnx=mv, label_mvnx=lab, time_s=grid)
    out.with_suffix(".json").write_text(json.dumps(checks, default=float))
    return mv, lab, grid, checks


# Control: la lectura del MVNX crudo reproduce la exportación en un ensayo incluido del mismo sujeto (si hay MVNX crudos)
REFERENCE = "Subj_07_MMH_Bim_08kg_Trial_1.mvnx"
if MVNX_DIR.exists() and not all((RECOVER_DIR / f.replace(".mvnx", ".npz")).exists() for f in RECOVER):
    base = SYNC_DIR / "bimanual"
    r = pd.read_csv(base / "trials.csv").set_index("mvnx_file").loc[REFERENCE]
    pos = {c["name"]: i for i, c in enumerate(json.loads((base / "columns.json").read_text())["mvnx"])}
    sl = slice(r.offset, r.offset + r.n_steps)
    with np.load(base / "series.npz") as z:
        ref_mv, ref_lab, ref_grid = z["mvnx"][sl][:, [pos[c] for c in RAW_COLUMNS]], z["label_mvnx"][sl], z["time_s"][sl]
    mv, lab, _, _ = process_mvnx(REFERENCE, grid=ref_grid)
    assert np.abs(mv - ref_mv).max() < 1e-3 and (lab == ref_lab).all(), "La lectura del MVNX crudo no reproduce la exportación"
    del ref_mv, mv

# Se conservan sólo los recuperados que pasan los mismos controles que el resto del MVNX
recovered = {f: recovered_trial(f) for f in RECOVER}
RECOVERED = [f for f, (_, _, _, c) in recovered.items() if c["n_time_gaps"] == 0 and c["labels_ok"] and c["nan"] == 0]
print(f"Ensayos recuperados que pasan los controles del MVNX: {len(RECOVERED)} de {len(RECOVER)} ({', '.join(RECOVERED)})")

In [ ]:
# Serie completa: todos los ensayos, uno tras otro. `trials` dice qué filas corresponden a cada ensayo.
rows, mv_parts, lab_parts, offsets = [], [], [], [0]
for conjunto in ("un_brazo", "bimanual"):
    # Exportación sincronizada de cada conjunto de ensayos: series a 12 Hz y tabla de ensayos
    base = SYNC_DIR / conjunto
    names = [c["name"] for c in json.loads((base / "columns.json").read_text())["mvnx"]]
    pos = {n: i for i, n in enumerate(names)}
    with np.load(base / "series.npz") as z:
        mvnx, label_mvnx = z["mvnx"][:, [pos[c] for c in RAW_COLUMNS]], z["label_mvnx"]
    for r in pd.read_csv(base / "trials.csv").itertuples(index=False):
        sl = slice(r.offset, r.offset + r.n_steps)
        mv, FEATURES = derive(mvnx[sl])                                # 782 columnas -> 30 variables
        rows.append({"trial_idx": len(rows), "mvnx_file": r.mvnx_file, "conjunto": conjunto, "subject": r.subject,
                     "offset": offsets[-1], "n_steps": r.n_steps, "recuperado": False})
        mv_parts.append(mv); lab_parts.append(label_mvnx[sl])
        offsets.append(offsets[-1] + r.n_steps)
    del mvnx, label_mvnx
# Se agregan al final los ensayos recuperados de la sección anterior
for f in RECOVERED:
    mv_raw, lab, grid, _ = recovered[f]
    mv, FEATURES = derive(mv_raw)
    rows.append({"trial_idx": len(rows), "mvnx_file": f, "conjunto": "bimanual", "subject": int(MVNX_RE.fullmatch(f)["subject"]),
                 "offset": offsets[-1], "n_steps": len(grid), "recuperado": True})
    mv_parts.append(mv); lab_parts.append(lab.astype(np.int64))
    offsets.append(offsets[-1] + len(grid))

trials = pd.DataFrame(rows)
offsets = np.asarray(offsets[:-1])                             # fila donde empieza cada ensayo
signals_raw = np.concatenate(mv_parts).astype(np.float32)      # (instantes, 30), unidades físicas
labels_series = np.concatenate(lab_parts).astype(np.int64)     # (instantes,), índice de LABELS o NO_LABEL
CHANNELS, N_CH = list(FEATURES), len(FEATURES)
BINARY = [i for i, n in enumerate(CHANNELS) if feature_unit(n) == "0/1"]   # contactos: el escalador no los modifica
del mv_parts, lab_parts
assert N_CH == 30 and not np.isnan(signals_raw).any() and trials["mvnx_file"].is_unique

print(f"{len(trials)} ensayos de {trials['subject'].nunique()} sujetos | {len(signals_raw):,} instantes "
      f"({len(signals_raw) / TARGET_FS / 3600:.1f} h) | {N_CH} variables")

In [ ]:
# Índices de ventanas: no se copian los datos, sólo se guarda dónde empieza cada ventana
def build_indices():
    sup, cpc = [], []
    for r in trials.itertuples():
        if r.n_steps < WIN:                                            # ensayo más corto que una ventana
            continue
        lab = labels_series[r.offset:r.offset + r.n_steps]
        # Ventanas sin etiqueta para CPC: una cada CPC_HOP pasos (2 s)
        cpc += [(r.trial_idx, r.subject, s) for s in range(0, r.n_steps - WIN + 1, CPC_HOP)]
        # Ventanas etiquetadas: una cada SUP_HOP pasos (0.5 s), con la etiqueta del instante final; sin etiqueta se descartan
        ends = np.arange(WIN - 1, r.n_steps, SUP_HOP)
        sup += [(r.trial_idx, r.subject, e - WIN + 1, e, lab[e]) for e in ends[lab[ends] != NO_LABEL]]
    return (pd.DataFrame(sup, columns=["trial_idx", "subject", "start_step", "end_step", "target"]),
            pd.DataFrame(cpc, columns=["trial_idx", "subject", "start_step"]))


def starts_of(frame):
    """Fila de `signals_raw` donde empieza cada ventana."""
    return offsets[frame["trial_idx"].to_numpy()] + frame["start_step"].to_numpy()


sup_index, cpc_index = build_indices()
sup_cv = sup_index[sup_index["subject"].isin(CV_SUBJECTS)]     # ventanas etiquetadas de los sujetos 1–12
cpc_cv = cpc_index[cpc_index["subject"].isin(CV_SUBJECTS)]     # ventanas sin etiqueta para CPC
assert not sup_cv["subject"].isin(TEST_SUBJECTS).any() and sup_cv["target"].between(0, len(LABELS) - 1).all()

print(f"Ventanas etiquetadas (sujetos 1–12): {len(sup_cv):,} | ventanas sin etiqueta para CPC: {len(cpc_cv):,}")



# Entrada del Random Forest: 5 estadísticas de cada variable en la ventana
STATS = ("media", "desv", "min", "max", "pendiente")
T_CENTERED = np.arange(WIN) - (WIN - 1) / 2                   # tiempo centrado, para calcular la pendiente


def window_stats(starts, chunk=8_192):
    """Las 5 estadísticas de cada variable en cada ventana -> (ventanas, 150), en unidades físicas."""
    view = sliding_window_view(signals_raw, WIN, axis=0)          # (instantes − WIN + 1, 30, WIN), sin copiar
    out = []
    for i in range(0, len(starts), chunk):                        # por bloques, para no llenar la memoria
        w = view[starts[i:i + chunk]].astype(np.float64)
        slope = (w * T_CENTERED).sum(axis=2) / (T_CENTERED ** 2).sum() * TARGET_FS   # pendiente por mínimos cuadrados, por segundo
        out.append(np.concatenate([w.mean(axis=2), w.std(axis=2), w.min(axis=2), w.max(axis=2), slope], axis=1))
    return np.concatenate(out).astype(np.float32)


# Escalador de las redes: se ajusta en cada ronda sólo con los sujetos de entrenamiento (sin fuga del sujeto evaluado)
def fit_scaler(train_subjects, seed=SEED):
    """Escalador robusto ajustado con los instantes de los sujetos de entrenamiento. Contactos (0/1): sin cambio."""
    rows = trials[trials["subject"].isin(train_subjects)]
    idx = np.concatenate([np.arange(r.offset, r.offset + r.n_steps) for r in rows.itertuples()])
    sample = signals_raw[np.random.default_rng(seed).choice(idx, min(SCALER_FIT_POINTS, len(idx)), replace=False)].astype(np.float64)
    # Recorte en los cuantiles 0.1 % y 99.9 %, luego (x − mediana) / (q95 − q5)
    lower, upper = np.quantile(sample, 0.001, axis=0), np.quantile(sample, 0.999, axis=0)
    x = np.clip(sample, lower, upper)
    q5, center, q95 = np.percentile(x, [5, 50, 95], axis=0)
    scale = np.where(q95 - q5 > 0, q95 - q5, 1.0)
    lower[BINARY], upper[BINARY], center[BINARY], scale[BINARY] = 0.0, 1.0, 0.0, 1.0
    return {k: v.astype(np.float32).tolist() for k, v in {"lower": lower, "upper": upper, "center": center, "scale": scale}.items()}


def apply_scaler(params, x=None):
    x = signals_raw if x is None else x
    lower, upper, center, scale = (np.asarray(params[k], dtype=np.float32) for k in ("lower", "upper", "center", "scale"))
    return ((np.clip(x, lower, upper) - center) / scale).astype(np.float32)

In [ ]:
# Ventanas etiquetadas de los sujetos reservados (sup_cv sólo tiene los sujetos 1–12)
sup_test = sup_index[sup_index["subject"].isin(TEST_SUBJECTS)]
assert set(sup_test["subject"]) == set(TEST_SUBJECTS), "Faltan ventanas de los sujetos 13 o 14"
display(sup_index.assign(actividad=sup_index["target"].map(dict(enumerate(LABELS))))
        .pivot_table(index="subject", columns="actividad", values="target", aggfunc="size", fill_value=0)[LABELS]
        .loc[list(CV_SUBJECTS) + list(TEST_SUBJECTS)].assign(total=lambda d: d.sum(axis=1)))

# 3. Subconjunto `EML`

Codos, manos y piernas: 10 variables, 11 sensores (notebook 11). La tabla del Random Forest usa sólo las 50 estadísticas de esas variables.

In [ ]:
SENSORS_ALL = ("Pelvis", "T8", "Head", "LeftShoulder", "RightShoulder", "LeftUpperArm", "RightUpperArm", "LeftForeArm",
               "RightForeArm", "LeftHand", "RightHand", "LeftUpperLeg", "RightUpperLeg", "LeftLowerLeg", "RightLowerLeg",
               "LeftFoot", "RightFoot")
both = lambda *names: [f"{lado}_{n}" for n in names for lado in ("izq", "der")]   # izquierda y derecha siempre juntas
pair = lambda *segments: [f"{side}{s}" for s in segments for side in ("Left", "Right")]

# Código -> nombre, variables y sensores que necesita. El orden es el de la tabla.
GROUPS = {
    "G": {"nombre": "Posiciones globales", "variables": ["pelvis_altura", "cabeza_altura"] + both("mano_altura", "mano_alcance"),
          "sensores": list(SENSORS_ALL)},
    "P": {"nombre": "Pelvis", "variables": ["pelvis_vel_horizontal", "pelvis_vel_vertical", "pelvis_giro"],
          "sensores": ["Pelvis"]},
    "T": {"nombre": "Tronco", "variables": ["tronco_flexion", "tronco_vel_angular"], "sensores": ["Pelvis", "T8"]},
    "C": {"nombre": "Cabeza", "variables": ["cuello_flexion"], "sensores": ["T8", "Head"]},
    "H": {"nombre": "Hombros", "variables": both("hombro_flexion", "hombro_abduccion"),
          "sensores": pair("Shoulder", "UpperArm")},
    "E": {"nombre": "Codos", "variables": both("codo_flexion"), "sensores": pair("UpperArm", "ForeArm")},
    "M": {"nombre": "Manos", "variables": both("muneca_flexion", "mano_rapidez"), "sensores": pair("ForeArm", "Hand")},
    "L": {"nombre": "Piernas", "variables": both("cadera_flexion", "rodilla_flexion"),
          "sensores": ["Pelvis"] + pair("UpperLeg", "LowerLeg")},
    "F": {"nombre": "Pies", "variables": both("talon", "punta"), "sensores": pair("LowerLeg", "Foot")},
}
# Control: los grupos reparten las 30 variables sin repetir ninguna, y sólo usan sensores del traje
grouped = [v for g in GROUPS.values() for v in g["variables"]]
assert sorted(grouped) == sorted(CHANNELS) and len(grouped) == len(set(grouped)), "Los grupos no cubren las 30 variables"
assert all(set(g["sensores"]) <= set(SENSORS_ALL) for g in GROUPS.values())


def subset_key(groups):
    """Nombre corto de un subconjunto: los códigos de sus grupos en el orden de GROUPS (p. ej. 'PTEM')."""
    return "".join(g for g in GROUPS if g in groups)


def variables_of(groups):
    """Variables de un subconjunto, en el orden de CHANNELS."""
    keep = {v for g in groups for v in GROUPS[g]["variables"]}
    return [c for c in CHANNELS if c in keep]


def sensors_of(groups):
    return sorted({s for g in groups for s in GROUPS[g]["sensores"]}, key=SENSORS_ALL.index)



SUBSET = ("E", "M", "L")                 # codos, manos y piernas: el mínimo del notebook 11
VARS = variables_of(SUBSET)
print(f"{subset_key(SUBSET)}: {len(VARS)} variables: {', '.join(VARS)}")

In [ ]:
TAB_COLS = [f"{s}_{n}" for s in STATS for n in VARS]       # 5 estadísticas × 10 variables
sup_all = pd.concat([sup_cv, sup_test])
X_rf = pd.DataFrame(window_stats(starts_of(sup_all)), index=sup_all.index, columns=[f"{s}_{n}" for s in STATS for n in CHANNELS])[TAB_COLS]
print(f"Random Forest -> {X_rf.shape[1]} características | {len(X_rf):,} ventanas (sujetos 1–14)")

# 4. Modelo y entrenamiento

El modelo, el preentrenamiento CPC (con parada temprana por una sonda lineal en el sujeto de validación) y el LP-FT son los del notebook 11, tomados del notebook 14 sin cambios. Sólo se usa la cabeza lineal.

In [ ]:
HEADS = ("lineal",)
MLP_HIDDEN, MLP_DROPOUT = 128, 0.3    # no se usan: la celda del modelo los necesita para definir make_head

In [ ]:
@dataclass
class CPCConfig:
    n_channels: int = 30
    enc_channels: tuple = (32, 64, 128)
    enc_kernel_size: int = 3
    enc_dropout: float = 0.2
    ar_hidden_size: int = 256
    ar_num_layers: int = 2
    ar_dropout: float = 0.2
    n_anchors: int = N_ANCHORS
    horizons: tuple = HORIZONS

    @property
    def z_dim(self):
        return self.enc_channels[-1]

    @property
    def c_dim(self):
        return self.ar_hidden_size


class CausalEncoder(nn.Module):
    """g_enc causal: 3 bloques Conv1d -> ReLU -> Dropout, relleno sólo a la izquierda. x: (B, canales, T) -> z: (B, 128, T)"""

    def __init__(self, cfg):
        super().__init__()
        k = cfg.enc_kernel_size
        self.left_pad = k - 1
        in_channels = (cfg.n_channels,) + cfg.enc_channels[:-1]
        self.convs = nn.ModuleList([nn.Conv1d(c_in, c_out, kernel_size=k) for c_in, c_out in zip(in_channels, cfg.enc_channels)])
        self.dropout = nn.Dropout(cfg.enc_dropout)

    def forward(self, x):
        z = x
        for conv in self.convs:
            z = self.dropout(F.relu(conv(F.pad(z, (self.left_pad, 0)))))
        return z


class CPCModel(nn.Module):
    """CPC con g_ar GRU: encoder + GRU + una W_k por horizonte; pérdida InfoNCE con negativos del batch."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = CausalEncoder(cfg)
        self.ar = nn.GRU(cfg.z_dim, cfg.c_dim, num_layers=cfg.ar_num_layers, dropout=cfg.ar_dropout, batch_first=True)
        self.heads = nn.ModuleList([nn.Linear(cfg.c_dim, cfg.z_dim, bias=False) for _ in cfg.horizons])

    def encode(self, x):
        """x: (B, canales, T) -> z: (B, T, 128), c: (B, T, 256)"""
        z = self.encoder(x).transpose(1, 2)
        return z, self.ar(z)[0]

    def info_nce(self, x):
        z, c = self.encode(x)
        lo, hi = max(1, z.shape[1] // 8), z.shape[1] - max(self.cfg.horizons)   # deja sitio para leer z_{t+12}
        target = torch.arange(len(x), device=x.device)
        losses = []
        for t in torch.randint(lo, hi, size=(self.cfg.n_anchors,)).tolist():
            for head, k in zip(self.heads, self.cfg.horizons):
                logits = head(c[:, t]) @ z[:, t + k].t()                         # (B, B): la diagonal es el positivo
                losses.append(F.cross_entropy(logits, target))
        return torch.stack(losses).mean()


def make_head(kind, dim, n_classes=len(LABELS)):
    """Cabeza de clasificación sobre c_T. 'lineal': la de los notebooks 10 y 11. 'mlp': una capa oculta no lineal."""
    if kind == "lineal":
        return nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.2), nn.Linear(dim, n_classes))
    if kind == "mlp":
        return nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, MLP_HIDDEN), nn.GELU(), nn.Dropout(MLP_DROPOUT),
                             nn.Linear(MLP_HIDDEN, n_classes))
    raise ValueError(kind)


class CPCClassifier(nn.Module):
    """Encoder y GRU de CPC + cabeza (lineal o MLP) sobre c_T, el contexto del último instante."""

    def __init__(self, cpc_model, head="lineal", n_classes=len(LABELS)):
        super().__init__()
        self.encoder, self.ar = cpc_model.encoder, cpc_model.ar
        self.head = make_head(head, cpc_model.cfg.c_dim, n_classes)

    def features(self, x):
        return self.ar(self.encoder(x).transpose(1, 2))[0][:, -1]

    def forward(self, x):
        return self.head(self.features(x))


def build_cpc(n_channels, seed=SEED):
    torch.manual_seed(seed)
    return CPCModel(CPCConfig(n_channels=n_channels)).to(DEVICE)

In [ ]:
# ----------------------------- Serie escalada de la ronda, sólo con las variables del subconjunto -----------------------------
S_T, WIN_RANGE = None, torch.arange(WIN, device=DEVICE)


def get_windows(starts):
    """Inicios de ventana -> x: (ventanas, variables del subconjunto, 48)."""
    idx = torch.as_tensor(np.asarray(starts), device=DEVICE)[:, None] + WIN_RANGE
    return S_T[:, idx].permute(1, 0, 2)


@torch.no_grad()
def batched(fn, starts, batch=512):
    return torch.cat([fn(get_windows(starts[s:s + batch])) for s in range(0, len(starts), batch)])


@torch.no_grad()
def predict(model, starts):
    model.eval()
    return batched(model, starts).argmax(1).cpu().numpy()


def class_weights(y):
    counts = np.bincount(y, minlength=len(LABELS))
    return torch.tensor(counts.sum() / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)


# ----------------------------- Preentrenamiento CPC (sin etiquetas) -----------------------------
class SameSubjectSampler:
    """Batches de CPC: ventanas al azar de UN solo sujeto, así los negativos de InfoNCE son de la misma persona."""

    def __init__(self, subjects, seed):
        self.rng = np.random.default_rng(seed)
        rows = [r for r in trials[trials["subject"].isin(subjects)].itertuples() if r.n_steps >= WIN]
        starts = np.concatenate([np.arange(r.offset, r.offset + r.n_steps - WIN + 1) for r in rows])
        subj = np.concatenate([np.full(r.n_steps - WIN + 1, r.subject) for r in rows])
        self.by_subject = {s: starts[subj == s] for s in np.unique(subj)}
        sizes = np.array([len(v) for v in self.by_subject.values()], dtype=float)
        self.p = sizes / sizes.sum()                                         # sujetos con más datos salen más seguido

    def batch(self):
        pool = self.by_subject[self.rng.choice(list(self.by_subject), p=self.p)]
        return self.rng.choice(pool, CPC_BATCH, replace=len(pool) < CPC_BATCH)


def probe_val_f1(model, rnd):
    """F1 en la validación interna de una sonda lineal (C = 1) sobre c_T. Decide la parada temprana de CPC."""
    model.eval()
    rep = lambda part: batched(lambda x: model.encode(x)[1][:, -1], rnd["sets"][part]["starts"]).float().cpu().numpy()
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    probe.fit(rep("probe_train"), rnd["sets"]["probe_train"]["y"])
    return f1_score(rnd["sets"]["val"]["y"], probe.predict(rep("val")), average="macro")


def train_cpc(rnd, seed=SEED):
    t0 = time.time()
    model = build_cpc(rnd["n_channels"], seed)
    opt = torch.optim.Adam(model.parameters(), lr=CPC_LR)
    sampler = SameSubjectSampler(rnd["train_subjects"], seed)
    best, bad = {"f1": -1.0, "epoch": 0, "state": copy.deepcopy(model.state_dict())}, 0
    for epoch in range(1, CPC_EPOCHS + 1):
        model.train()
        for _ in range(rnd["n_steps"]):
            loss = model.info_nce(get_windows(sampler.batch()))             # pérdida contrastiva: no usa etiquetas
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        if epoch % CPC_EVAL_EVERY == 0 or epoch == CPC_EPOCHS:
            f1_val = probe_val_f1(model, rnd)
            if f1_val > best["f1"] + 1e-4:
                best, bad = {"f1": f1_val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
            else:
                bad += 1
        if bad >= CPC_PATIENCE:
            break
    model.load_state_dict(best["state"])
    return model, {"best_epoch": best["epoch"], "f1_val": best["f1"], "minutes": (time.time() - t0) / 60}


# ----------------------------- LP-FT (con etiquetas) -----------------------------
def linear_probe_head(model, sets, seed):
    """Etapa LP: entrena sólo la cabeza con el encoder y la GRU congelados."""
    model.eval()
    X_tr, X_val = batched(model.features, sets["ft_train"]["starts"]), batched(model.features, sets["val"]["starts"])
    y_tr = torch.as_tensor(np.array(sets["ft_train"]["y"]), device=DEVICE)
    weights = class_weights(sets["ft_train"]["y"])
    opt = torch.optim.AdamW(model.head.parameters(), lr=LP_LR, weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    best, bad = {"f1": -1.0, "state": copy.deepcopy(model.head.state_dict())}, 0
    for epoch in range(1, LP_EPOCHS + 1):
        model.head.train()
        order = rng.permutation(len(y_tr))
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = torch.as_tensor(order[s:s + FINETUNE_BATCH], device=DEVICE)
            loss = F.cross_entropy(model.head(X_tr[idx]), y_tr[idx], weight=weights)
            opt.zero_grad(); loss.backward(); opt.step()
        model.head.eval()
        with torch.no_grad():
            f1_val = f1_score(sets["val"]["y"], model.head(X_val).argmax(1).cpu().numpy(), average="macro")
        if f1_val > best["f1"] + 1e-4:
            best, bad = {"f1": f1_val, "state": copy.deepcopy(model.head.state_dict())}, 0
        else:
            bad += 1
        if bad >= LP_PATIENCE:
            break
    model.head.load_state_dict(best["state"])
    return best["f1"]


def finetune(cpc_model, sets, head="lineal", seed=SEED):
    """LP-FT: la cabeza de LP es el punto de partida (época 0) y después se ajusta toda la red."""
    t0 = time.time()
    torch.manual_seed(seed)
    model = CPCClassifier(cpc_model, head=head).to(DEVICE)
    best = {"f1": linear_probe_head(model, sets, seed), "epoch": 0, "state": copy.deepcopy(model.state_dict())}
    opt = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()) + list(model.ar.parameters()), "lr": FINETUNE_LR_BACKBONE},
        {"params": model.head.parameters(), "lr": FINETUNE_LR_HEAD}], weight_decay=1e-4)
    train = sets["ft_train"]
    weights = class_weights(train["y"])
    y_all = torch.as_tensor(np.array(train["y"]), device=DEVICE)
    rng, bad = np.random.default_rng(seed), 0
    for epoch in range(1, FT_EPOCHS + 1):
        model.train()
        order = rng.permutation(len(train["y"]))
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = order[s:s + FINETUNE_BATCH]
            loss = F.cross_entropy(model(get_windows(train["starts"][idx])), y_all[idx], weight=weights)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        f1_val = f1_score(sets["val"]["y"], predict(model, sets["val"]["starts"]), average="macro")
        if f1_val > best["f1"] + 1e-4:
            best, bad = {"f1": f1_val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
        if bad >= FT_PATIENCE:
            break
    model.load_state_dict(best["state"])
    return model, {"best_epoch": best["epoch"], "f1_val": best["f1"], "minutes": (time.time() - t0) / 60}

# 5. Entrenamiento y evaluación

Por escenario y semilla:

1. **Escalador** ajustado sólo con los sujetos de entrenamiento del escenario.
2. **CPC sin etiquetas** con los sujetos de entrenamiento; parada temprana con la sonda lineal en el sujeto 12.
3. **LP-FT** con las etiquetas de los sujetos de entrenamiento; parada temprana con el F1 del sujeto 12.
4. **Random Forest** con las ventanas etiquetadas de los mismos sujetos de entrenamiento.
5. **Prueba:** F1 macro, predicciones y probabilidades en cada sujeto de prueba.

Cada corrida se guarda en `<escenario>/semilla<k>.json` (y los pesos de la red en `.pt`). Si el notebook se vuelve a ejecutar, se cargan.

In [ ]:
def prepare_scenario(key):
    """Escalador y serie EML en el dispositivo, y conjuntos de ventanas del escenario."""
    global S_T
    sc = SCENARIOS[key]
    scaler = fit_scaler(sc["train"])
    cols = [CHANNELS.index(v) for v in VARS]
    S_T = torch.from_numpy(np.ascontiguousarray(apply_scaler(scaler)[:, cols].T)).to(DEVICE)
    train = sup_all[sup_all["subject"].isin(sc["train"])]
    sets = {"probe_train": eval_set(train, PROBE_TRAIN_MAX), "ft_train": eval_set(train, FT_TRAIN_CAP),
            "val": eval_set(sup_all[sup_all["subject"] == sc["val"]]),
            **{f"test_{s}": eval_set(sup_all[sup_all["subject"] == s]) for s in sc["test"]}}
    n_steps = max(1, len(cpc_index[cpc_index["subject"].isin(sc["train"])]) // CPC_BATCH)
    assert not set(sc["train"]) & set(sc["test"]) and sc["val"] not in sc["train"] + sc["test"]
    return {"train_subjects": sc["train"], "sets": sets, "n_channels": len(cols),
            "n_steps": min(n_steps, STEPS_CAP) if STEPS_CAP else n_steps, "scaler": scaler}


def eval_set(frame, cap=None):
    """Ventanas de un conjunto. Con `cap`, una muestra estratificada por actividad."""
    if cap is not None and len(frame) > cap:
        frame = frame.groupby("target", group_keys=False).sample(frac=cap / len(frame), random_state=SEED)
    return {"index": frame.index.to_numpy(), "starts": starts_of(frame), "y": frame["target"].to_numpy()}


@torch.no_grad()
def predict_proba(model, starts):
    model.eval()
    return batched(lambda x: F.softmax(model(x), dim=1), starts).cpu().numpy()


def run(key, seed, rnd):
    """CPC-GRU y Random Forest de un escenario y una semilla -> resultados por sujeto de prueba."""
    path = RUN_DIR / key / f"semilla{seed}.json"
    if path.exists():
        return json.loads(path.read_text())
    sc, out = SCENARIOS[key], {"escenario": key, "semilla": seed}
    tests = [f"test_{s}" for s in sc["test"]]

    # --- CPC-GRU: preentrenamiento sin etiquetas + LP-FT ---
    cpc_model, info_cpc = train_cpc(rnd, seed)
    model, info_ft = finetune(cpc_model, rnd["sets"], head="lineal", seed=seed)
    out["gru"] = {"cpc": info_cpc, "ajuste_fino": info_ft}
    for t in tests:
        proba = predict_proba(model, rnd["sets"][t]["starts"])
        out["gru"][t] = {"f1": f1_score(rnd["sets"][t]["y"], proba.argmax(1), average="macro"), "proba": proba.round(5).tolist()}
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"cpc": cpc_model.state_dict(), "ajuste_fino": model.state_dict(), "variables": VARS,
                "scaler": rnd["scaler"]}, path.with_suffix(".pt"))
    del cpc_model, model

    # --- Random Forest con los mismos sujetos de entrenamiento ---
    t0 = time.time()
    train = sup_all["subject"].isin(sc["train"]).to_numpy()
    rf = RandomForestClassifier(**RF_PARAMS, random_state=seed).fit(X_rf[train].to_numpy(), sup_all["target"].to_numpy()[train])
    out["rf"] = {"minutes": (time.time() - t0) / 60}
    for t in tests:
        s = int(t.split("_")[1])
        mask = (sup_all["subject"] == s).to_numpy()
        proba = np.zeros((mask.sum(), len(LABELS)))
        proba[:, rf.classes_] = rf.predict_proba(X_rf[mask].to_numpy())
        out["rf"][t] = {"f1": f1_score(sup_all["target"].to_numpy()[mask], proba.argmax(1), average="macro"),
                        "proba": proba.round(5).tolist()}
    del rf

    out["y"] = {t: rnd["sets"][t]["y"].tolist() for t in tests}
    path.write_text(json.dumps(out, default=float))
    f1s = " | ".join(f"sujeto {t[5:]}: GRU {out['gru'][t]['f1']:.3f} · RF {out['rf'][t]['f1']:.3f}" for t in tests)
    log(f"{sc['nombre']} · semilla {seed} | {f1s} | CPC época {info_cpc['best_epoch']} · FT época {info_ft['best_epoch']} | "
        f"{info_cpc['minutes'] + info_ft['minutes'] + out['rf']['minutes']:.1f} min")
    return out

In [ ]:
results = {}
for key in SCENARIOS:                                  # A primero; B después, porque entrena con el sujeto 13
    rnd = None
    for seed in SEEDS:
        if not (RUN_DIR / key / f"semilla{seed}.json").exists() and rnd is None:
            rnd = prepare_scenario(key)
        results[(key, seed)] = run(key, seed, rnd)

# 6. Resultados

## 6.1 F1 macro en cada sujeto de prueba

Por modelo y sujeto: F1 de cada semilla, media ± desviación entre semillas y F1 del **ensamble** (promedio de las probabilidades de las 3 semillas).

In [ ]:
MODELS = {"gru": "CPC-GRU EML", "rf": "Random Forest EML"}


def ensemble_pred(key, model, t):
    return np.mean([np.asarray(results[(key, s)][model][t]["proba"]) for s in SEEDS], axis=0).argmax(1)


rows = []
for key, sc in SCENARIOS.items():
    for s in sc["test"]:
        t = f"test_{s}"
        y = np.asarray(results[(key, SEEDS[0])]["y"][t])
        for m, name in MODELS.items():
            f1 = np.array([results[(key, seed)][m][t]["f1"] for seed in SEEDS])
            rows.append({"escenario": sc["nombre"], "sujeto": s, "modelo": name,
                         **{f"semilla {seed}": v for seed, v in zip(SEEDS, f1)},
                         "media": f1.mean(), "desv.": f1.std(ddof=1) if len(f1) > 1 else 0.0,
                         "ensamble": f1_score(y, ensemble_pred(key, m, t), average="macro")})
final = pd.DataFrame(rows).set_index(["escenario", "sujeto", "modelo"])
display(final.style.format("{:.3f}"))

## 6.2 ¿Coincide con lo que estimó LOSO?

Si el LOSO de los sujetos 1–12 era una estimación honesta, el F1 de los sujetos 13 y 14 debe caer dentro del rango que mostró (notebook 11, `EML`, una semilla, 10 sujetos de entrenamiento). Se marca en qué percentil de los 12 sujetos de LOSO cae cada sujeto de prueba.

In [ ]:
loso = {"gru": np.array([json.loads((NB11_DIR / "gru_ft" / "EML" / f"ronda_s{s:02d}.json").read_text())["f1"] for s in CV_SUBJECTS]),
        "rf": np.array([json.loads((NB11_DIR / "rf" / "EML.json").read_text())["f1"][str(s)] for s in CV_SUBJECTS])}

rows = []
for m, name in MODELS.items():
    row = {"modelo": name, "LOSO media": loso[m].mean(), "LOSO desv.": loso[m].std(ddof=1),
           "LOSO peor": loso[m].min(), "LOSO mejor": loso[m].max()}
    for s in SCENARIOS["A_final"]["test"]:
        v = final.loc[(SCENARIOS["A_final"]["nombre"], s, name), "media"]
        row[f"sujeto {s}"] = v
        row[f"percentil sujeto {s}"] = f"{(loso[m] < v).mean():.0%}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("modelo").style.format(
    {c: "{:.3f}" for c in ["LOSO media", "LOSO desv.", "LOSO peor", "LOSO mejor", "sujeto 13", "sujeto 14"]}))

# Gráfica: F1 de LOSO por sujeto (puntos grises) y de los sujetos de prueba (colores), por modelo
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (m, name) in enumerate(MODELS.items()):
    ax.scatter(np.full(len(loso[m]), i) + np.random.default_rng(0).uniform(-0.08, 0.08, len(loso[m])), loso[m],
               color="#b8b8b8", s=30, label="LOSO, sujetos 1–12" if i == 0 else None)
    ax.hlines(loso[m].mean(), i - 0.2, i + 0.2, color="#777", lw=1.5)
    for s, color in zip(SCENARIOS["A_final"]["test"], ("#2a6fdb", "#d4572a")):
        f1 = [results[("A_final", seed)][m][f"test_{s}"]["f1"] for seed in SEEDS]
        ax.scatter(np.full(len(f1), i + (0.3 if s == 13 else 0.4)), f1, color=color, s=40, marker="D",
                   label=f"sujeto {s} (cada semilla)" if i == 0 else None)
ax.set_xticks(range(len(MODELS)), list(MODELS.values()))
ax.set_ylabel("F1 macro")
ax.set_title("Escenario A: sujetos de prueba frente al rango de LOSO")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6.3 CPC-GRU frente a Random Forest

Diferencia de F1 (CPC-GRU − Random Forest) en cada sujeto de prueba, por semilla y con el ensamble. Con sólo dos sujetos no hay prueba estadística: se reporta la diferencia y su signo en cada semilla.

In [ ]:
rows = []
for key, sc in SCENARIOS.items():
    for s in sc["test"]:
        t = f"test_{s}"
        d = np.array([results[(key, seed)]["gru"][t]["f1"] - results[(key, seed)]["rf"][t]["f1"] for seed in SEEDS])
        idx = (sc["nombre"], s)
        rows.append({"escenario": sc["nombre"], "sujeto": s, "diferencia media": d.mean(),
                     "semillas a favor de CPC-GRU": f"{(d > 0).sum()} de {len(d)}",
                     "diferencia del ensamble": final.loc[idx + ("CPC-GRU EML",), "ensamble"] - final.loc[idx + ("Random Forest EML",), "ensamble"]})
display(pd.DataFrame(rows).set_index(["escenario", "sujeto"]).style.format(
    {"diferencia media": "{:+.3f}", "diferencia del ensamble": "{:+.3f}"}))

## 6.4 F1 por actividad y matrices de confusión (escenario A, ensamble)

In [ ]:
rows = {}
for s in SCENARIOS["A_final"]["test"]:
    t = f"test_{s}"
    y = np.asarray(results[("A_final", SEEDS[0])]["y"][t])
    for m, name in MODELS.items():
        rows[(f"sujeto {s}", name)] = f1_score(y, ensemble_pred("A_final", m, t), average=None, labels=range(len(LABELS)))
display(pd.DataFrame(rows, index=LABELS).T.style.format("{:.3f}"))

fig, axes = plt.subplots(len(SCENARIOS["A_final"]["test"]), len(MODELS), figsize=(10, 9))
for r, s in enumerate(SCENARIOS["A_final"]["test"]):
    t = f"test_{s}"
    y = np.asarray(results[("A_final", SEEDS[0])]["y"][t])
    for c, (m, name) in enumerate(MODELS.items()):
        cm = confusion_matrix(y, ensemble_pred("A_final", m, t), labels=range(len(LABELS)), normalize="true")
        ax = axes[r, c]
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(len(LABELS)):
            for j in range(len(LABELS)):
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if cm[i, j] > 0.6 else "#333")
        ax.set_xticks(range(len(LABELS)), LABELS)
        ax.set_yticks(range(len(LABELS)), LABELS)
        ax.set_title(f"Sujeto {s} · {name}", fontsize=10)
        ax.set_xlabel("predicción")
        ax.set_ylabel("real")
plt.tight_layout()
plt.show()

## 6.5 Escenario B: comparación con Bassani et al. (2025)

Sujeto 14 con 12 sujetos de entrenamiento (1–11 y 13) más el 12 de validación, frente a lo que reportan con 13 sujetos de entrenamiento. Sus cifras salen de la tabla 5 y la tabla 4 del artículo; el F1 del BiLSTM aparece como 0.903 en el resumen y la sección 4.3, y como 0.906 en la tabla 5 (ver [comparacion_bassani_2025.md](../documentacion/comparacion_bassani_2025.md)).

**Lo que no cambia respecto a nuestro protocolo:** ventanas de 4 s a 12 Hz etiquetadas con el instante final, variables `EML` sin sEMG y pesos de clase en lugar de réplicas con ruido. Sólo se replica la partición de sujetos.

In [ ]:
BASSANI = {"BiLSTM (Bassani)": {"F1": 0.903, "N": 0.891, "LT": 0.945, "PT": 0.840, "LF": 0.903, "PF": 0.888, "K": 0.923, "W": 0.934},
           "RCNN (Bassani)": {"F1": 0.892, "N": 0.901, "LT": 0.924, "PT": 0.851, "LF": 0.887, "PF": 0.867, "K": 0.914, "W": 0.902},
           "DeepConvLSTM (Bassani)": {"F1": 0.903}}
t = "test_14"
y = np.asarray(results[("B_bassani", SEEDS[0])]["y"][t])
rows = dict(BASSANI)
for m, name in MODELS.items():
    pred = ensemble_pred("B_bassani", m, t)
    rows[f"{name} (nuestro, ensamble)"] = {"F1": f1_score(y, pred, average="macro"),
                                           **dict(zip(LABELS, f1_score(y, pred, average=None, labels=range(len(LABELS)))))}
    rows[f"{name} (nuestro, media de semillas)"] = {"F1": final.loc[(SCENARIOS["B_bassani"]["nombre"], 14, name), "media"]}
display(pd.DataFrame(rows).T[["F1"] + LABELS].style.format("{:.3f}", na_rep="—"))

# Cuánto cambia el sujeto 14 al agregar el sujeto 13 al entrenamiento (escenario B frente a A)
for m, name in MODELS.items():
    a = final.loc[(SCENARIOS["A_final"]["nombre"], 14, name), "media"]
    b = final.loc[(SCENARIOS["B_bassani"]["nombre"], 14, name), "media"]
    print(f"{name}: sujeto 14 con 11 sujetos de entrenamiento {a:.3f} → con 12 {b:.3f} ({b - a:+.3f})")

## 6.6 Detalles del entrenamiento

Época elegida de CPC (sonda en el sujeto 12) y del ajuste fino (0 = se quedó con la cabeza de la etapa LP), F1 de validación y tiempos.

In [ ]:
rows = []
for (key, seed), r in results.items():
    rows.append({"escenario": SCENARIOS[key]["nombre"], "semilla": seed,
                 "época CPC": r["gru"]["cpc"]["best_epoch"], "F1 sonda (val.)": r["gru"]["cpc"]["f1_val"],
                 "época FT": r["gru"]["ajuste_fino"]["best_epoch"], "F1 FT (val.)": r["gru"]["ajuste_fino"]["f1_val"],
                 "min CPC": r["gru"]["cpc"]["minutes"], "min LP-FT": r["gru"]["ajuste_fino"]["minutes"], "min RF": r["rf"]["minutes"]})
display(pd.DataFrame(rows).set_index(["escenario", "semilla"]).style.format(
    {"F1 sonda (val.)": "{:.3f}", "F1 FT (val.)": "{:.3f}", "min CPC": "{:.1f}", "min LP-FT": "{:.1f}", "min RF": "{:.1f}"}))

# 7. Conclusiones

*Se completan después de ejecutar el notebook.*